# Day 02: Endüstriyel Veri Modelleri ve Şema Doğrulama (Pydantic v2)

**Merinos Industrial AI Internship Portfolio — Day 02**  
**Yazar:** Seydi Eryılmaz (@seydivakkas)  
**Lisans:** Özel Lisans — Tüm Hakları Saklıdır (All Rights Reserved)  

---

## 1. Problem Tanımı ve Mühendislik Motivasyonu

Endüstriyel yapay zeka sistemlerinde, görüntü işleme ve derin öğrenme modellerine beslenen verilerin kalitesi ve tutarlılığı en kritik başarı faktörüdür. Merinos üretim tesislerinde her gün binlerce desen görseli taranmakta, teknik şartnameler oluşturulmakta ve kalite kontrol istasyonlarında kamera sistemlerinden metadata üretilmektedir.

Geleneksel Python sözlükleri (`dict`) veya gevşek tipleme (duck typing) şu sorunlara yol açar:
- **Sessiz Veri Bozulması (Silent Data Corruption):** Negatif piksel boyutları, geçersiz renk kanalları (örn. 5 kanal) veya eksik alanlar inference anında patlar.
- **Tip Güvensizliği:** String olarak gelen sayısal değerler (`"300"` yerine `300`) matematiksel hesaplamalarda `TypeError` üretir.
- **Sözleşme Eksikliği (Lack of API Contracts):** Frontend, veri tabanı ve derin öğrenme servisleri arasında JSON şema uyumsuzlukları oluşur.

Bu çalışmada, **Pydantic v2** kullanarak endüstriyel halı ürünleri, görüntü metadata'sı, teknik dokümanlar ve AI inference istek/yanıtları için katı, derleme/çalışma zamanında doğrulanabilen veri modelleri tasarlıyoruz.

## 2. Neden Önemli? (Endüstriyel Etki & İş Değeri)

- **Hatalı Üretimi Önleme:** Hatalı çözünürlük veya en-boy oranı ile modele beslenen bir desen, kesim ve dokuma tezgahlarında telafisi imkansız hammadde israfına yol açar.
- **Sistemler Arası Entegrasyon (Interoperability):** Üretim Takip Sistemi (MES), Kurumsal Kaynak Planlama (ERP) ve Vektör Veritabanı (Qdrant/Milvus) tek bir doğrulanmış JSON şema üzerinden haberleşir.
- **Yüksek Performans:** Pydantic v2'nin Rust çekirdeği (`pydantic-core`), v1'e göre 5x-20x daha hızlı doğrulama sunarak yüksek frekanslı kamera hatlarında gecikme (latency) yaratmaz.

## 3. Matematiksel ve Kavramsal Temeller

Bir halının geometrisi ve dijital temsili katı fiziksel sınırlarla çevrilidir:

1. **En-Boy Oranı (Aspect Ratio, $\alpha$):**
$$\alpha = \frac{L}{W}$$
burada $L$ uzunluk (cm), $W$ genişlik (cm)'dir. Endüstriyel standart bir halı için $0.3 \le \alpha \le 5.0$ olmalıdır. Bu aralık dışındaki değerler boru hattına giren anomali veridir.

2. **Piksel Çözünürlüğü ve Yoğunluk (DPI):**
$$DPI = \frac{\text{Piksel Sayısı}}{\text{Fiziksel Boyut (inç)}}$$
Dokuma kalite kontrolünde optik taramanın minimum $DPI \ge 72$ (ideal olarak $300$) olması şarttır.

3. **Renk Doğrulama (Hex Triplet):**
$$C_{hex} = \#RRGGBB, \quad R, G, B \in \{00, \dots, FF\}$$
Hex kodunun tam 7 karakter ve geçerli onaltılık sayı olması gerekir.

## 4. Kütüphane ve Araç İncelemesi: Pydantic v2 vs Alternatives

| Özellik | Standart `dict` | `dataclasses` | `Pydantic v1` | `Pydantic v2` | `attrs` |
|---|---|---|---|---|---|
| Çekirdek Motor | CPython | CPython | Python | **Rust (pydantic-core)** | CPython |
| Otomatik Tip Coercion | ❌ Yok | ❌ Yok | ✅ Var | ✅ **Var (Katı/Gevşek mod)** | Kısmi |
| JSON Schema Üretimi | ❌ Yok | ❌ Yok | ✅ Var | ✅ **Var (OpenAPI 3.1)** | ❌ Ek kütüphane |
| Derin Validasyon | ❌ Manuel | ❌ `__post_init__` | ✅ Python AST | ✅ **Rust C-Level** | Kısmi |
| Doğrulama Hızı | N/A | Düşük | 1x (Referans) | **5x - 17x Daha Hızlı** | 2x - 3x |

In [1]:
import sys
from pathlib import Path

# Dinamik kök dizin ekleme
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.startswith('day') else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 5. Minimal Çalışır Kod: Ortam ve Modellerin Tanımlanması
from __future__ import annotations
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field, field_validator, model_validator, ConfigDict, ValidationError

class ColorSpaceEnum(str, Enum):
    RGB = "RGB"
    BGR = "BGR"
    GRAYSCALE = "GRAYSCALE"
    LAB = "LAB"

class MaterialEnum(str, Enum):
    WOOL = "WOOL"
    ACRYLIC = "ACRYLIC"
    POLYESTER = "POLYESTER"
    BAMBOO_SILK = "BAMBOO_SILK"

class CarpetDimensions(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)
    
    width_cm: float = Field(..., gt=0.0, le=1200.0, description="Halı genişliği (cm)")
    length_cm: float = Field(..., gt=0.0, le=1200.0, description="Halı uzunluğu (cm)")
    pile_height_mm: float = Field(..., gt=0.0, le=50.0, description="Hav yüksekliği (mm)")
    
    @property
    def aspect_ratio(self) -> float:
        return round(self.length_cm / self.width_cm, 3)
        
    @model_validator(mode="after")
    def validate_aspect_ratio_sanity(self) -> CarpetDimensions:
        ratio = self.length_cm / self.width_cm
        if ratio < 0.3 or ratio > 5.0:
            raise ValueError(f"Anormal en-boy oranı: {ratio:.2f}. Boy/en oranı [0.3, 5.0] arasında olmalıdır.")
        return self

# Test örnekleme
dim = CarpetDimensions(width_cm=160.0, length_cm=230.0, pile_height_mm=11.0)
print(f"Doğrulandı: {dim.width_cm}x{dim.length_cm} cm, Hav: {dim.pile_height_mm} mm, Aspect Ratio: {dim.aspect_ratio}")

Doğrulandı: 160.0x230.0 cm, Hav: 11.0 mm, Aspect Ratio: 1.438


In [2]:
# Gelişmiş Endüstriyel Ürün Modeli Tanımı
import re

class CarpetProduct(BaseModel):
    model_config = ConfigDict(extra="forbid", validate_assignment=True)
    
    product_id: str = Field(..., description="Benzersiz Merinos ürün kodu: MER-XXX-XXXX")
    name: str = Field(..., min_length=2, max_length=120)
    collection: str = Field(..., min_length=2, max_length=60)
    material: MaterialEnum
    dimensions: CarpetDimensions
    dominant_hex_palette: List[str] = Field(..., min_length=1, max_length=10)
    
    @field_validator("product_id")
    @classmethod
    def validate_product_id_format(cls, v: str) -> str:
        pattern = r"^MER-[A-Z0-9]{3,8}-[0-9]{3,5}$"
        if not re.match(pattern, v):
            raise ValueError(f"Geçersiz ürün ID formatı '{v}'. Beklenen: 'MER-<KOLEKSIYON>-<NUMARA>' (Örn: MER-PRST-1002)")
        return v
        
    @field_validator("dominant_hex_palette")
    @classmethod
    def validate_hex_colors(cls, colors: List[str]) -> List[str]:
        hex_pattern = r"^#[0-9a-fA-F]{6}$"
        for c in colors:
            if not re.match(hex_pattern, c):
                raise ValueError(f"Geçersiz hex renk kodu: '{c}'. Beklenen: '#RRGGBB'")
        return colors

product = CarpetProduct(
    product_id="MER-PRST-1002",
    name="Prestij Venedik Dokuma",
    collection="Prestij",
    material=MaterialEnum.ACRYLIC,
    dimensions=dim,
    dominant_hex_palette=["#2B3A42", "#4F6D7A", "#C0D6DF", "#E8DAB2"]
)
print("Ürün Modeli Başarıyla Oluşturuldu:")
print(product.model_dump_json(indent=2))

Ürün Modeli Başarıyla Oluşturuldu:
{
  "product_id": "MER-PRST-1002",
  "name": "Prestij Venedik Dokuma",
  "collection": "Prestij",
  "material": "ACRYLIC",
  "dimensions": {
    "width_cm": 160.0,
    "length_cm": 230.0,
    "pile_height_mm": 11.0
  },
  "dominant_hex_palette": [
    "#2B3A42",
    "#4F6D7A",
    "#C0D6DF",
    "#E8DAB2"
  ]
}


## 6. Deneyler ve Performans Analizi

Pydantic v2'nin Rust çekirdeğinin performansını standart Python sözlük oluşturma ve `dataclasses` ile karşılaştıralım.

In [3]:
import timeit
from dataclasses import dataclass

@dataclass
class RawCarpet:
    width: float
    length: float
    pile: float

N = 20_000

# 1. Standart dict
dict_time = timeit.timeit(
    lambda: {"width_cm": 160.0, "length_cm": 230.0, "pile_height_mm": 11.0},
    number=N
)

# 2. Dataclass
dc_time = timeit.timeit(
    lambda: RawCarpet(160.0, 230.0, 11.0),
    number=N
)

# 3. Pydantic v2 (tam doğrulama, Rust motoru)
pydantic_time = timeit.timeit(
    lambda: CarpetDimensions(width_cm=160.0, length_cm=230.0, pile_height_mm=11.0),
    number=N
)

print(f"{N} Nesne İçin İşlem Süreleri:")
print(f"- Standard dict (Doğrulamasız)   : {dict_time * 1000:.2f} ms ({N/dict_time:,.0f} ops/s)")
print(f"- Dataclass (Doğrulamasız)       : {dc_time * 1000:.2f} ms ({N/dc_time:,.0f} ops/s)")
print(f"- Pydantic v2 (Katı Doğrulamalı) : {pydantic_time * 1000:.2f} ms ({N/pydantic_time:,.0f} ops/s)")
print(f"\nSonuç: Pydantic v2, tam katı validasyon kuralı uygularken saniyede yüzbinlerce nesne doğrulayabilir.")

20000 Nesne İçin İşlem Süreleri:
- Standard dict (Doğrulamasız)   : 1.84 ms (10,871,338 ops/s)
- Dataclass (Doğrulamasız)       : 2.51 ms (7,959,565 ops/s)
- Pydantic v2 (Katı Doğrulamalı) : 29.41 ms (680,069 ops/s)

Sonuç: Pydantic v2, tam katı validasyon kuralı uygularken saniyede yüzbinlerce nesne doğrulayabilir.


## 7. Görselleştirme ve JSON Schema Yapısı

Pydantic modellerinin en büyük avantajı, doğrudan OpenAPI / JSON Schema spesifikasyonuna uygun şemalar üretebilmesidir. Bu şemalar web servislerinde Swagger UI ve frontend doğrulamasında otomatik kullanılır.

In [4]:
# JSON Schema Çıkarma ve İnceleme
schema = CarpetProduct.model_json_schema()
print("CarpetProduct JSON Schema Anahtarları:", list(schema.keys()))
print("Zorunlu Alanlar:", schema.get("required"))
print("Özellikler (Properties):")
for prop, details in schema.get("properties", {}).items():
    print(f"  - {prop}: {details.get('type', details.get('$ref', 'complex'))}")

CarpetProduct JSON Schema Anahtarları: ['$defs', 'additionalProperties', 'properties', 'required', 'title', 'type']
Zorunlu Alanlar: ['product_id', 'name', 'collection', 'material', 'dimensions', 'dominant_hex_palette']
Özellikler (Properties):
  - product_id: string
  - name: string
  - collection: string
  - material: #/$defs/MaterialEnum
  - dimensions: #/$defs/CarpetDimensions
  - dominant_hex_palette: array


## 8. Doğrulama ve Testler

Modellerimizin sınır koşullarını ve doğrulama mantığını test edelim.

In [5]:
# Başarılı senaryo testi
assert product.dimensions.width_cm == 160.0
assert product.material == MaterialEnum.ACRYLIC
assert len(product.dominant_hex_palette) == 4

# JSON serileştirme ve geri okuma (Roundtrip) testi
json_str = product.model_dump_json()
loaded_product = CarpetProduct.model_validate_json(json_str)
assert loaded_product.product_id == product.product_id
assert loaded_product.dimensions.length_cm == product.dimensions.length_cm
print("✅ Tüm roundtrip serileştirme testleri başarıyla geçti!")

✅ Tüm roundtrip serileştirme testleri başarıyla geçti!


## 9. Hata Durumları ve Uç Senaryolar (Failure Cases & Edge Cases)

Sisteme giren geçersiz verilerin sessizce kabul edilmeyip nasıl yakalandığını simüle edelim.

In [6]:
failure_scenarios = [
    (
        "Geçersiz Halı Boyutu (Negatif En)",
        lambda: CarpetDimensions(width_cm=-50.0, length_cm=200.0, pile_height_mm=10.0)
    ),
    (
        "Anormal En-Boy Oranı (Aşırı Dar ve Uzun Şerit)",
        lambda: CarpetDimensions(width_cm=20.0, length_cm=600.0, pile_height_mm=10.0)
    ),
    (
        "Hatalı Ürün ID Şablonu (Yanlış Format)",
        lambda: CarpetProduct(
            product_id="INVALID-ID-123",
            name="Test",
            collection="Test",
            material=MaterialEnum.WOOL,
            dimensions=dim,
            dominant_hex_palette=["#FFFFFF"]
        )
    ),
    (
        "Geçersiz Hex Rengi (#GGRRBB)",
        lambda: CarpetProduct(
            product_id="MER-USAK-001",
            name="Test",
            collection="Test",
            material=MaterialEnum.WOOL,
            dimensions=dim,
            dominant_hex_palette=["#GG0011"]
        )
    )
]

print("Hata Yakalama Testleri Simülasyonu:\n")
for title, scenario in failure_scenarios:
    try:
        scenario()
        print(f"❌ BAŞARISIZ: '{title}' hatası yakalanmadı!")
    except (ValidationError, ValueError) as e:
        print(f"✅ DOĞRU YAKALANDI: [{title}] -> Hata detayı:")
        # İlk hata mesajını yazdır
        first_err = str(e).split("\n")[0]
        print(f"   {first_err}\n")

Hata Yakalama Testleri Simülasyonu:

✅ DOĞRU YAKALANDI: [Geçersiz Halı Boyutu (Negatif En)] -> Hata detayı:
   1 validation error for CarpetDimensions

✅ DOĞRU YAKALANDI: [Anormal En-Boy Oranı (Aşırı Dar ve Uzun Şerit)] -> Hata detayı:
   1 validation error for CarpetDimensions

✅ DOĞRU YAKALANDI: [Hatalı Ürün ID Şablonu (Yanlış Format)] -> Hata detayı:
   1 validation error for CarpetProduct

✅ DOĞRU YAKALANDI: [Geçersiz Hex Rengi (#GGRRBB)] -> Hata detayı:
   1 validation error for CarpetProduct



## 10. Mühendislik Çıkarımları ve Sonraki Adım

### Çıkarımlar:
1. **Veri Boru Hattının İlk Savunma Hattı:** Modeller, anomali verilerin boru hattının derinliklerine (vektörleştirme, fine-tuning veya veri tabanı) girmesini giriş kapısında engeller.
2. **Schema-Driven Architecture:** Pydantic v2 modelleri tek gerçeklik kaynağıdır (Single Source of Truth). FastAPI endpointleri, veritabanı ORM modelleri ve Qdrant payload yapıları bu modellerden türetilir.
3. **İmmutability:** Teknik şartnameler ve dokümanlar `frozen=True` ile güvenceye alınarak runtime sırasında istem dışı değişiklikler engellenir.

### Sonraki Adım (Day 03):
Day 03'te, bu veri modellerini kullanarak çok kaynaklı (CSV, JSON, metadata logları) veri işleme hattı kuracağız; Pandas normalizer ve parser algoritmalarıyla ham üretim verilerini doğrulanmış modellere dönüştüreceğiz.